# IMDB Simple RNN — Prediction and Inference

This notebook demonstrates how to load the trained **Simple RNN sentiment model**, reproduce the required IMDB preprocessing, and make predictions on new movie reviews.

### What we will do

1. Load the IMDB vocabulary and trained model
2. Inspect the model architecture and weights
3. Decode an IMDB integer sequence back to text
4. Preprocess new user text into the same integer representation expected by the model
5. Apply sequence padding
6. Generate a sentiment prediction
7. Interpret the sigmoid output as the probability of the positive class

> **Important:** Inference preprocessing must remain consistent with the preprocessing used during training.

## 1. Import Required Libraries

In [7]:
import re
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

## 2. Define Inference Configuration

The model was trained with a fixed vocabulary limit and a maximum sequence length. These values must match the training configuration during inference.

In [8]:
VOCAB_SIZE = 10_000
MAX_SEQUENCE_LENGTH = 500
MODEL_NAME = "simple_rnn_imdb.h5"

## 3. Locate and Load the Trained Model

The trained model is stored in the repository's `models/` directory. The fallback paths below make the notebook easier to run from either the repository root or the `notebooks/` directory.

In [9]:
candidate_paths = [
    Path.cwd() / "models" / MODEL_NAME,
    Path.cwd().parent / "models" / MODEL_NAME,
    Path(MODEL_NAME),
]

MODEL_PATH = next((path for path in candidate_paths if path.exists()), None)

if MODEL_PATH is None:
    raise FileNotFoundError(
        f"Could not find {MODEL_NAME}. Expected it in a models/ directory or the current directory."
    )

print(f"Loading model from: {MODEL_PATH}")
model = load_model(MODEL_PATH)

Loading model from: e:\imdb-movie-review-sentiment-analysis-simple-rnn\models\simple_rnn_imdb.h5


In [10]:
# Load the IMDB dataset word index
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

## 4. Inspect the Loaded Model

In [11]:
# Load the pre-trained model with ReLU activation
model = load_model('simple_rnn_imdb.h5')
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (32, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [12]:
model.get_weights()

[array([[-1.09077   ,  1.0350633 ,  0.02392277, ...,  0.21703486,
          0.9186467 , -0.08250786],
        [-0.02079579,  0.07625301, -0.01921176, ...,  0.05926689,
          0.07409113,  0.04320552],
        [-0.05687032,  0.17954342,  0.03881063, ..., -0.00164656,
          0.01298865,  0.0954427 ],
        ...,
        [-0.02314784, -0.01816396, -0.0206836 , ..., -0.02333968,
         -0.07094826,  0.08474475],
        [ 0.04964662,  0.05343529,  0.00725298, ...,  0.06244898,
         -0.01867161, -0.10127337],
        [ 0.13523474, -0.06278417, -0.01692815, ..., -0.11892289,
         -0.10444389,  0.05454928]], shape=(10000, 128), dtype=float32),
 array([[-0.08185606, -0.08008295,  0.08454056, ...,  0.1395195 ,
         -0.15019985,  0.09528898],
        [-0.03997448,  0.09773041, -0.02217148, ...,  0.00764158,
         -0.07458631,  0.02136598],
        [-0.18537107,  0.10862887,  0.05317311, ...,  0.07412732,
         -0.16286203,  0.06602841],
        ...,
        [-0.0437481

### Inspect Weight Shapes

`get_weights()` returns the numerical parameters learned during training. Instead of printing every value, we inspect the shapes so the notebook remains readable.

In [13]:
for idx, weights in enumerate(model.get_weights(), start=1):
    print(f"Weight tensor {idx}: {weights.shape}")

Weight tensor 1: (10000, 128)
Weight tensor 2: (128, 128)
Weight tensor 3: (128, 128)
Weight tensor 4: (128,)
Weight tensor 5: (128, 1)
Weight tensor 6: (1,)


## 5. Load the IMDB Vocabulary

The IMDB dataset provides a mapping from words to integer token IDs. During inference, we use the same vocabulary mapping to convert a new review into the representation expected by the model.

In [14]:
word_index = imdb.get_word_index()
reverse_word_index = {index: word for word, index in word_index.items()}

print(f"Vocabulary entries available: {len(word_index):,}")

Vocabulary entries available: 88,584


## 6. Decode an Integer-Encoded Review

The IMDB dataset stores reviews as integer sequences. This helper converts those integers back into readable text for inspection.

The IMDB format reserves low integer IDs for special tokens, so the word-index lookup uses the same offset convention as the dataset.

In [15]:
def decode_review(encoded_review):
    return " ".join(
        reverse_word_index.get(token_id - 3, "?")
        for token_id in encoded_review
        if token_id >= 4
    )

## 7. Inspect a Sample Encoded Review

This step is only for understanding the dataset representation. It is not required for making a new prediction.

In [16]:
(_, _), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

sample_encoded_review = x_test[0]
sample_label = y_test[0]

print("Encoded review (first 20 tokens):")
print(sample_encoded_review[:20])
print(f"\nEncoded review length: {len(sample_encoded_review)}")
print(
    f"Label: {sample_label} ({'Positive' if sample_label == 1 else 'Negative'})")

Encoded review (first 20 tokens):
[1, 591, 202, 14, 31, 6, 717, 10, 10, 2, 2, 5, 4, 360, 7, 4, 177, 5760, 394, 354]

Encoded review length: 68
Label: 0 (Negative)


In [17]:
decoded_review = decode_review(sample_encoded_review)
print(decoded_review)

please give this one a miss br br and the rest of the cast rendered terrible performances the show is flat flat flat br br i don't know how michael madison could have allowed this one on his plate he almost seemed to know this wasn't going to work out and his performance was quite so all you madison fans give this a miss


## 8. Preprocess New User Input

A new review is plain text, so it must be converted into the same integer-token format used by the trained model.

The preprocessing below: 

- converts text to lowercase
- removes punctuation so tokens such as `fantastic!` can match the vocabulary entry `fantastic`
- maps unknown or out-of-vocabulary words to the IMDB unknown-token ID
- applies the `+3` offset used by the IMDB encoding convention

In [18]:
def preprocess_text(text, vocab_size=VOCAB_SIZE, max_length=MAX_SEQUENCE_LENGTH):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Review must be a non-empty string.")

    # Normalize text in a way that is closer to the IMDB tokenization convention.
    normalized_text = re.sub(r"[^a-z0-9\s]", " ", text.lower())
    words = normalized_text.split()

    encoded_review = []
    for word in words:
        index = word_index.get(word, 2)

        # Match the training-time vocabulary limit: words outside the top VOCAB_SIZE
        # are treated as unknown tokens.
        if index >= vocab_size:
            index = 2

        encoded_review.append(index + 3)

    return sequence.pad_sequences(
        [encoded_review],
        maxlen=max_length,
        padding="pre"
    )

## 9. Inspect Preprocessed Input

The model expects a 2D batch input with shape `(batch_size, sequence_length)`. Using one review produces a batch size of `1`.

In [19]:
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

preprocessed_input = preprocess_text(example_review)

print("Original review:")
print(example_review)
print(f"\nPreprocessed shape: {preprocessed_input.shape}")
print("Encoded sequence (last 20 tokens):")
print(preprocessed_input[0, -20:])

Original review:
This movie was fantastic! The acting was great and the plot was thrilling.

Preprocessed shape: (1, 500)
Encoded sequence (last 20 tokens):
[   0    0    0    0    0    0    0   14   20   16  777    4  116   16
   87    5    4  114   16 3017]


## 10. Define the Sentiment Prediction Function

The final layer uses a sigmoid activation, so the model output is a value between `0` and `1`. In this binary classifier, a value above `0.5` is interpreted as **Positive** and otherwise **Negative**.

In [20]:
def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction = model.predict(preprocessed_input, verbose=0)

    positive_probability = float(prediction[0][0])
    sentiment = "Positive" if positive_probability >= 0.5 else "Negative"

    return sentiment, positive_probability

## 11. Make a Prediction

In [21]:
sentiment, positive_probability = predict_sentiment(example_review)

print(f"Review: {example_review}")
print(f"Sentiment: {sentiment}")
print(f"Positive-class probability: {positive_probability:.4f}")
print(f"Positive-class probability (%): {positive_probability * 100:.2f}%")

Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Negative
Positive-class probability: 0.4177
Positive-class probability (%): 41.77%


## 12. Try Additional Reviews

Use the examples below to test the trained model on different inputs.

In [22]:
test_reviews = [
    "The movie was excellent and I really enjoyed every scene.",
    "The movie was boring, poorly acted, and disappointing.",
    "The story was interesting, but the ending was disappointing."
]

for review in test_reviews:
    sentiment, score = predict_sentiment(review)
    print(f"Review: {review}")
    print(f"Sentiment: {sentiment}")
    print(f"Positive probability: {score:.4f}")
    print("-" * 80)

Review: The movie was excellent and I really enjoyed every scene.
Sentiment: Positive
Positive probability: 0.8692
--------------------------------------------------------------------------------
Review: The movie was boring, poorly acted, and disappointing.
Sentiment: Negative
Positive probability: 0.0220
--------------------------------------------------------------------------------
Review: The story was interesting, but the ending was disappointing.
Sentiment: Negative
Positive probability: 0.0860
--------------------------------------------------------------------------------


## 13. Key Takeaways

- The trained H5 model contains the architecture and learned weights required for inference.
- New text must be converted into the same integer-token representation expected by the model.
- The sequence must be padded to the training length of `500` tokens.
- The prediction is a sigmoid probability for the **Positive** class.
- A threshold of `0.5` is used to convert the probability into a Positive/Negative label.
- Keeping training and inference preprocessing consistent is essential for reliable predictions.
- The same preprocessing and prediction logic can later be reused by the Streamlit application.

In [23]:
# Step 2: Helper Functions
# Function to decode reviews
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess user input


def preprocess_text(text):
    words = text.lower().split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review

In [24]:
# Prediction  function

def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)

    prediction = model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'

    return sentiment, prediction[0][0]

In [25]:
# Step 4: User Input and Prediction
# Example review for prediction
example_review = "This movie was fantastic! The acting was great and the plot was thrilling."

sentiment, score = predict_sentiment(example_review)

print(f'Review: {example_review}')
print(f'Sentiment: {sentiment}')
print(f'Prediction Score: {score}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Review: This movie was fantastic! The acting was great and the plot was thrilling.
Sentiment: Negative
Prediction Score: 0.197520449757576
